# El Niño Shoreline Project: Model Comparison with a Locked Test Set

This notebook repeats the model comparison from `shoreline_model_comparison_simplified.ipynb`,
with one change: **about 20% of the data is locked away before anything else happens.**

For each test set:
1. **Lock away** the test data (the models never see it until step 4).
2. **Cross-validate on the other 80%** with the same method as the main notebook
   (leave-one-winter-out for Question 1, 10 folds grouped by coastal section for Question 2).
3. **Train** each model on all of the 80%.
4. **Test once** on the locked data.

To check that the conclusions don't depend on *which* data we lock away, we try several test sets:

| | Test sets tried |
|---|---|
| **Question 1** (new winter) | A. 4 winters spread across La Niña → El Niño · B. the last 4 winters · C. 4 random winters |
| **Question 2** (new coastline) | A. 22 random coastal sections · B. every 5th section along the coast |

The models, features, settings and GBT tuning are the same as in the main notebook.
Drop-column feature importance (8c) is not repeated here.

**This notebook is slow (likely 1–2 hours).** Each step prints its progress, so you can leave it running.
Everything this notebook saves goes to `models/locked_test/`. The main notebook and the other files in `models/` are not touched.

## 1. Setup (same settings as the main notebook)

In [1]:
import json
import time
from pathlib import Path

import joblib
import sklearn

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

ROOT_DIR  = Path.cwd().parent           # this notebook lives in scripts/
DATA_PATH = ROOT_DIR / "processed" / "final_modeling_dataset.csv"
OUT_DIR   = ROOT_DIR / "models" / "locked_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "shoreline_change_m"           # Mar–Apr average position minus Oct–Nov average position
OUTLIER_LIMIT_M = 50                    # drop rows where |change| is above this

BEACH_FEATURES = ["BeachWidth_m", "beach_slope", "shore_orientation_sin", "shore_orientation_cos"]
WAVE_FEATURES  = ["winter_wave_power", "wave_dir_sin", "wave_dir_cos"]

FEATURE_SETS = {
    "beach":               BEACH_FEATURES,
    "beach + wave":        BEACH_FEATURES + WAVE_FEATURES,
    "beach + wave + RONI": BEACH_FEATURES + WAVE_FEATURES + ["RONI"],
}

## 2. Load the data and remove outliers (same as the main notebook)

In [2]:
df = pd.read_csv(DATA_PATH)
df = df[df[TARGET].abs() <= OUTLIER_LIMIT_M]
df = df.reset_index(drop=True)
df["section"] = df["transect_id"].str[:11]      # first 11 characters = coastal section

print("Rows:", len(df))
print("Winters:", df["winter"].nunique(), "| Coastal sections:", df["section"].nunique())

Rows: 60584
Winters: 21 | Coastal sections: 110


## 3. The models (copied from the main notebook)

In [3]:
class TransectAverage(BaseEstimator, RegressorMixin):
    """Baseline: predict each transect's average change in the training data."""
    def fit(self, X, y):
        training = pd.DataFrame({"transect_id": X["transect_id"].values, "change": np.asarray(y)})
        self.averages_ = training.groupby("transect_id")["change"].mean()
        self.overall_ = training["change"].mean()      # backup for transects never seen
        return self

    def predict(self, X):
        predictions = X["transect_id"].map(self.averages_)
        predictions = predictions.fillna(self.overall_)
        return predictions.to_numpy()


class TransectAveragePlus(BaseEstimator, RegressorMixin):
    """Transect average + a second model fit to each row's difference from its transect's average."""
    def __init__(self, model=None, features=None):
        self.model = model
        self.features = features

    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.means_ = y.groupby(X["transect_id"]).mean()
        self.overall_ = float(y.mean())
        leftovers = y - X["transect_id"].map(self.means_)
        self.model_ = clone(self.model).fit(X[self.features], leftovers)
        return self

    def predict(self, X):
        normal = X["transect_id"].map(self.means_).fillna(self.overall_).to_numpy()
        return normal + self.model_.predict(X[self.features])


def make_linear():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression())

def make_gam():
    return make_pipeline(SimpleImputer(strategy="median"), SplineTransformer(knots="quantile"), Ridge())

def make_rf():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(n_estimators=100, min_samples_leaf=50, max_features=0.5, n_jobs=-1, random_state=42),
    )

GBT_GRID = {"max_leaf_nodes": [15, 31], "min_samples_leaf": [100, 500]}

def make_gbt():
    base = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, early_stopping=True,
                                         validation_fraction=0.1, n_iter_no_change=20, random_state=42)
    return GridSearchCV(base, GBT_GRID, cv=GroupKFold(n_splits=3), scoring="neg_mean_absolute_error")

small_gbt = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                                          min_samples_leaf=500, random_state=42)

### The list of models for each question

Each entry is: name, model, features, and whether the model needs group labels (only the GBT, for its tuning).
Question 1 uses the same models as Test 1 in the main notebook. Question 2 uses the models from the main notebook's
Question 2 table (beach and beach + wave; RONI can't tell places apart).

In [4]:
def question_1_models():
    models = [
        ["Baseline: overall average",  DummyRegressor(strategy="mean"), ["RONI"],        False],
        ["Baseline: transect average", TransectAverage(),               ["transect_id"], False],
    ]
    for set_name, features in FEATURE_SETS.items():
        models.append([f"Linear: {set_name}", make_linear(), features, False])
        models.append([f"GAM: {set_name}",    make_gam(),    features, False])
        models.append([f"RF: {set_name}",     make_rf(),     features, False])
        models.append([f"GBT: {set_name}",    make_gbt(),    features, True])
    models.append(["Transect average + linear RONI",
                   TransectAveragePlus(LinearRegression(), ["RONI"]),
                   ["transect_id", "RONI"], False])
    models.append(["Transect average + GBT beach + wave (control)",
                   TransectAveragePlus(small_gbt, FEATURE_SETS["beach + wave"]),
                   ["transect_id"] + FEATURE_SETS["beach + wave"], False])
    models.append(["Transect average + GBT beach + wave + RONI",
                   TransectAveragePlus(small_gbt, FEATURE_SETS["beach + wave + RONI"]),
                   ["transect_id"] + FEATURE_SETS["beach + wave + RONI"], False])
    return models


def question_2_models():
    models = [["Baseline: overall average", DummyRegressor(strategy="mean"), ["RONI"], False]]
    for set_name in ["beach", "beach + wave"]:
        features = FEATURE_SETS[set_name]
        models.append([f"Linear: {set_name}", make_linear(), features, False])
        models.append([f"GAM: {set_name}",    make_gam(),    features, False])
        models.append([f"RF: {set_name}",     make_rf(),     features, False])
        models.append([f"GBT: {set_name}",    make_gbt(),    features, True])
    return models

## 4. The four steps, for one test set

`run_test_set` does steps 1–4 for every model and returns a table with two scores per model:
- **CV** = cross-validation on the 80% (used to compare models, like the main notebook)
- **Test** = one final check on the locked 20%

If `save_folder` is given, it also saves (for the live demo):
- every final model (trained on the 80%) as a `.pkl` file
- `test_rows.csv`: the locked test rows, with all features
- `test_predictions.csv`: each model's predictions for the locked test rows
- `model_info.json`: which features each model uses, and which data was locked away

In [5]:
def file_name(model_name):
    """Turn a model name into a file name, e.g. "GBT: beach + wave" -> "gbt_beach_wave.pkl"."""
    name = model_name.lower()
    for old, new in [(" + ", "_"), (": ", "_"), (" ", "_"), ("(", ""), (")", "")]:
        name = name.replace(old, new)
    return name + ".pkl"


def run_test_set(test_name, is_test, group_column, splitter, models, save_folder=None):
    # Step 1: lock away the test data
    dev  = df[~is_test]       # the 80%: used for cross-validation and training
    test = df[is_test]        # the 20%: not touched until step 4
    print(f"\n=== {test_name} ===")
    print(f"Training rows: {len(dev):,} | Locked test rows: {len(test):,}")

    rows = []
    saved_predictions = {}     # used only when save_folder is given
    model_info = {}
    for name, model, features, needs_groups in models:
        start = time.time()
        groups = dev[group_column]

        # Step 2: cross-validation on the 80% only
        if needs_groups:
            extra = {"groups": groups}        # the GBT's tuning keeps groups together too
        else:
            extra = None
        cv_predictions = cross_val_predict(model, dev[features], dev[TARGET],
                                           groups=groups, cv=splitter, params=extra)

        # Step 3: train on all of the 80%
        final_model = clone(model)
        if needs_groups:
            final_model.fit(dev[features], dev[TARGET], groups=groups)
        else:
            final_model.fit(dev[features], dev[TARGET])

        # Step 4: test once on the locked data
        test_predictions = final_model.predict(test[features])

        rows.append({
            "test set": test_name,
            "model": name,
            "CV R²":        r2_score(dev[TARGET], cv_predictions),
            "CV MAE (m)":   mean_absolute_error(dev[TARGET], cv_predictions),
            "Test R²":      r2_score(test[TARGET], test_predictions),
            "Test MAE (m)": mean_absolute_error(test[TARGET], test_predictions),
        })
        print(f"  {name:48s} done in {time.time() - start:5.0f} s")

        # Save the final model and its predictions (only for the test sets we demo)
        if save_folder is not None:
            if needs_groups:
                model_to_save = final_model.best_estimator_   # the GBT: keep only the winning settings
            else:
                model_to_save = final_model
            joblib.dump(model_to_save, save_folder / file_name(name), compress=3)
            saved_predictions[name] = test_predictions
            model_info[file_name(name)] = {"model": name, "features": features}

    if save_folder is not None:
        id_columns = ["transect_id", "section", "winter"]
        all_features = FEATURE_SETS["beach + wave + RONI"]
        test[id_columns + all_features + [TARGET]].to_csv(save_folder / "test_rows.csv", index=False)

        prediction_table = test[id_columns + ["RONI", TARGET]].copy()
        for name, predictions in saved_predictions.items():
            prediction_table[name] = predictions
        prediction_table.to_csv(save_folder / "test_predictions.csv", index=False)

        model_info["locked test set"] = test_name
        model_info["locked " + group_column + "s"] = sorted(test[group_column].unique().tolist())
        model_info["sklearn_version"] = sklearn.__version__
        with open(save_folder / "model_info.json", "w") as f:
            json.dump(model_info, f, indent=2)
        print(f"  Saved {len(saved_predictions)} models + test rows + predictions to {save_folder.relative_to(ROOT_DIR)}")

    return pd.DataFrame(rows)

---
# Question 1: Does El Niño (RONI) help predict a new winter?

## 5. Choose the locked test winters (about 20% = 4 of 21 winters)

- **A. Spread across El Niño / La Niña:** sort the winters by RONI and take 4 evenly spaced ones, including the
  strongest La Niña and the strongest El Niño.
- **B. Last 4 winters:** the most recent winters, like predicting the future.
- **C. 4 random winters:** picked at random with a fixed seed (42).

All three were chosen by rule, before seeing any results.

In [6]:
winter_roni = df.groupby("winter")["RONI"].first().sort_values()     # winters from La Niña to El Niño
all_winters = sorted(df["winter"].unique())                          # winters in time order

positions = np.linspace(0, len(winter_roni) - 1, 4).round().astype(int)   # 4 evenly spaced positions
rng = np.random.default_rng(42)

Q1_TEST_SETS = {
    "A: spread across El Niño/La Niña": list(winter_roni.index[positions]),
    "B: last 4 winters":                all_winters[-4:],
    "C: 4 random winters":              sorted(rng.choice(all_winters, size=4, replace=False).tolist()),
}

for test_name, winters in Q1_TEST_SETS.items():
    print(test_name)
    for winter in winters:
        print(f"    {winter}  RONI = {winter_roni[winter]:+.2f}")

A: spread across El Niño/La Niña
    2007-2008  RONI = -1.56
    2000-2001  RONI = -0.70
    2003-2004  RONI = +0.18
    2015-2016  RONI = +1.94
B: last 4 winters
    2017-2018  RONI = -1.02
    2018-2019  RONI = +0.70
    2019-2020  RONI = +0.14
    2020-2021  RONI = -1.30
C: 4 random winters
    2001-2002  RONI = -0.16
    2009-2010  RONI = +1.20
    2013-2014  RONI = -0.28
    2014-2015  RONI = +0.54


## 6. Run Question 1 (slow: the GBT tuning runs inside every one of the 17 folds)

Cross-validation on the 80% uses **leave-one-winter-out**, the same as Test 1 in the main notebook.

The models for **test set B (last 4 winters)** are saved to `models/locked_test/q1_last_4_winters/` for the live demo.

In [7]:
q1_tables = []
overall_start = time.time()
Q1_SAVE_FOLDER = OUT_DIR / "q1_last_4_winters"
Q1_SAVE_FOLDER.mkdir(parents=True, exist_ok=True)

for test_name, winters in Q1_TEST_SETS.items():
    is_test = df["winter"].isin(winters)

    # Save the models only for test set B (last 4 winters)
    if test_name.startswith("B"):
        save_folder = Q1_SAVE_FOLDER
    else:
        save_folder = None

    table = run_test_set(test_name, is_test, "winter", LeaveOneGroupOut(), question_1_models(), save_folder)
    q1_tables.append(table)

q1_results = pd.concat(q1_tables, ignore_index=True)
q1_results.to_csv(OUT_DIR / "q1_results.csv", index=False)
print(f"\nQuestion 1 finished in {(time.time() - overall_start) / 60:.0f} minutes. Saved to models/locked_test/q1_results.csv")


=== A: spread across El Niño/La Niña ===
Training rows: 49,525 | Locked test rows: 11,059
  Baseline: overall average                        done in     0 s
  Baseline: transect average                       done in     0 s
  Linear: beach                                    done in     0 s
  GAM: beach                                       done in     1 s
  RF: beach                                        done in     7 s
  GBT: beach                                       done in   829 s
  Linear: beach + wave                             done in     1 s
  GAM: beach + wave                                done in     2 s
  RF: beach + wave                                 done in    10 s
  GBT: beach + wave                                done in  1275 s
  Linear: beach + wave + RONI                      done in     1 s
  GAM: beach + wave + RONI                         done in     2 s
  RF: beach + wave + RONI                          done in    16 s
  GBT: beach + wave + RONI            

## 7. Question 1 results

### 7a. Test R² for every model, under each locked test set
Higher is better. The bar to beat is **Baseline: transect average**.

In [8]:
q1_results.pivot(index="model", columns="test set", values="Test R²").loc[q1_results["model"].unique()].round(3)

test set,A: spread across El Niño/La Niña,B: last 4 winters,C: 4 random winters
model,,,
Baseline: overall average,-0.016,-0.010,-0.000
Baseline: transect average,0.504,0.449,0.400
Linear: beach,0.011,0.034,0.036
GAM: beach,0.075,0.067,0.086
RF: beach,0.342,0.302,0.298
GBT: beach,0.344,0.306,0.299
Linear: beach + wave,0.042,0.054,0.058
GAM: beach + wave,0.106,0.082,0.112
RF: beach + wave,0.422,0.375,0.352


### 7b. Does adding RONI help? (change in Test R² from adding RONI)
Negative = RONI made predictions worse on the locked winters.

In [9]:
PAIRS = {
    "Transect average":       ("Baseline: transect average", "Transect average + linear RONI"),
    "Transect average + GBT": ("Transect average + GBT beach + wave (control)", "Transect average + GBT beach + wave + RONI"),
    "Linear":                 ("Linear: beach + wave", "Linear: beach + wave + RONI"),
    "GAM":                    ("GAM: beach + wave",    "GAM: beach + wave + RONI"),
    "Random forest":          ("RF: beach + wave",     "RF: beach + wave + RONI"),
    "GBT":                    ("GBT: beach + wave",    "GBT: beach + wave + RONI"),
}

test_r2 = q1_results.pivot(index="model", columns="test set", values="Test R²")
roni_effect = {}
for family, (without_roni, with_roni) in PAIRS.items():
    roni_effect[family] = test_r2.loc[with_roni] - test_r2.loc[without_roni]

pd.DataFrame(roni_effect).T.round(3)

test set,A: spread across El Niño/La Niña,B: last 4 winters,C: 4 random winters
Transect average,-0.015,0.001,0.002
Transect average + GBT,-0.142,-0.010,-0.009
Linear,-0.014,0.001,0.001
GAM,-0.101,0.043,0.090
Random forest,-0.255,-0.087,-0.022
GBT,-0.178,-0.029,-0.031


### 7c. Cross-validation (80%) vs. locked test (20%)
If the CV scores and the test scores tell the same story, the model choices made with cross-validation hold up on unseen winters.

In [10]:
q1_results.set_index(["test set", "model"]).round(3)

CV R²  \
test set                         model                                                  
A: spread across El Niño/La Niña Baseline: overall average                     -0.012   
                                 Baseline: transect average                     0.438   
                                 Linear: beach                                  0.025   
                                 GAM: beach                                     0.076   
                                 RF: beach                                      0.305   
                                 GBT: beach                                     0.312   
                                 Linear: beach + wave                           0.048   
                                 GAM: beach + wave                              0.097   
                                 RF: beach + wave                               0.370   
                                 GBT: beach + wave                              0.382   
                                 Linear: beach + wave + RONI                    0.032   
                                 GAM: beach + wave + RONI                       0.090   
                                 RF: beach + wave + RONI                        0.257   
                                 GBT: beach + wave + RONI                       0.319   
                                 Transect average + linear RONI                 0.422   
                                 Transect average + GBT beach + wave (control)  0.438   
                                 Transect average + GBT beach + wave + RONI     0.409   
B: last 4 winters                Baseline: overall average                     -0.009   
                                 Baseline: transect average                     0.451   
                                 Linear: beach                                  0.024   
                                 GAM: beach                                     0.083   
                                 RF: beach                                      0.315   
                                 GBT: beach                                     0.323   
                                 Linear: beach + wave                           0.049   
                                 GAM: beach + wave                              0.107   
                                 RF: beach + wave                               0.382   
                                 GBT: beach + wave                              0.395   
                                 Linear: beach + wave + RONI                    0.040   
                                 GAM: beach + wave + RONI                       0.074   
                                 RF: beach + wave + RONI                        0.241   
                                 GBT: beach + wave + RONI                       0.295   
                                 Transect average + linear RONI                 0.442   
                                 Transect average + GBT beach + wave (control)  0.451   
                                 Transect average + GBT beach + wave + RONI     0.374   
C: 4 random winters              Baseline: overall average                     -0.005   
                                 Baseline: transect average                     0.474   
                                 Linear: beach                                  0.030   
                                 GAM: beach                                     0.084   
                                 RF: beach                                      0.330   
                                 GBT: beach                                     0.339   
                                 Linear: beach + wave                           0.055   
                                 GAM: beach + wave                              0.106   
                                 RF: beach + wave                               0.400   
                                 GBT: beach + wave                         

---
# Question 2: Do beach and wave features explain shoreline change on new coastline?

## 8. Choose the locked test sections (about 20% = 22 of 110 sections)

- **A. 22 random sections:** picked at random with a fixed seed (42).
- **B. Every 5th section along the coast:** spread evenly from south to north.

In [11]:
all_sections = sorted(df["section"].unique())
rng = np.random.default_rng(42)

Q2_TEST_SETS = {
    "A: 22 random sections":  sorted(rng.choice(all_sections, size=22, replace=False).tolist()),
    "B: every 5th section":   all_sections[2::5],
}

for test_name, sections in Q2_TEST_SETS.items():
    print(f"{test_name} ({len(sections)} sections): {', '.join(sections)}")

A: 22 random sections (22 sections): usa_CA_0008, usa_CA_0009, usa_CA_0010, usa_CA_0014, usa_CA_0020, usa_CA_0049, usa_CA_0058, usa_CA_0062, usa_CA_0063, usa_CA_0064, usa_CA_0069, usa_CA_0076, usa_CA_0082, usa_CA_0089, usa_CA_0090, usa_CA_0094, usa_CA_0097, usa_CA_0098, usa_CA_0106, usa_CA_0109, usa_CA_0114, usa_CA_0126
B: every 5th section (22 sections): usa_CA_0003, usa_CA_0008, usa_CA_0013, usa_CA_0018, usa_CA_0025, usa_CA_0031, usa_CA_0038, usa_CA_0044, usa_CA_0052, usa_CA_0057, usa_CA_0062, usa_CA_0067, usa_CA_0072, usa_CA_0077, usa_CA_0088, usa_CA_0094, usa_CA_0099, usa_CA_0104, usa_CA_0109, usa_CA_0114, usa_CA_0119, usa_CA_0124


## 9. Run Question 2

Cross-validation on the 80% uses **10 folds grouped by coastal section**, the same as Test 2 in the main notebook.

The models for **test set A (22 random sections)** are saved to `models/locked_test/q2_22_random_sections/` for the live demo.

In [12]:
q2_tables = []
overall_start = time.time()
Q2_SAVE_FOLDER = OUT_DIR / "q2_22_random_sections"
Q2_SAVE_FOLDER.mkdir(parents=True, exist_ok=True)

for test_name, sections in Q2_TEST_SETS.items():
    is_test = df["section"].isin(sections)

    # Save the models only for test set A (22 random sections)
    if test_name.startswith("A"):
        save_folder = Q2_SAVE_FOLDER
    else:
        save_folder = None

    table = run_test_set(test_name, is_test, "section", GroupKFold(n_splits=10), question_2_models(), save_folder)
    q2_tables.append(table)

q2_results = pd.concat(q2_tables, ignore_index=True)
q2_results.to_csv(OUT_DIR / "q2_results.csv", index=False)
print(f"\nQuestion 2 finished in {(time.time() - overall_start) / 60:.0f} minutes. Saved to models/locked_test/q2_results.csv")


=== A: 22 random sections ===
Training rows: 43,927 | Locked test rows: 16,657
  Baseline: overall average                        done in     0 s
  Linear: beach                                    done in     0 s
  GAM: beach                                       done in     0 s
  RF: beach                                        done in     4 s
  GBT: beach                                       done in   511 s
  Linear: beach + wave                             done in     0 s
  GAM: beach + wave                                done in     1 s
  RF: beach + wave                                 done in     5 s
  GBT: beach + wave                                done in   519 s
  Saved 9 models + test rows + predictions to models/locked_test/q2_22_random_sections

=== B: every 5th section ===
Training rows: 50,680 | Locked test rows: 9,904
  Baseline: overall average                        done in     0 s
  Linear: beach                                    done in     0 s
  GAM: beach      

## 10. Question 2 results

### 10a. Test R² for every model, under each locked test set
Higher is better. The bar to beat is **Baseline: overall average** (an unseen beach has no history to average).

In [13]:
q2_results.pivot(index="model", columns="test set", values="Test R²").loc[q2_results["model"].unique()].round(3)

test set,A: 22 random sections,B: every 5th section
model,,
Baseline: overall average,-0.010,-0.000
Linear: beach,0.037,-0.034
GAM: beach,0.090,-0.017
RF: beach,-0.013,-0.077
GBT: beach,-0.002,-0.045
Linear: beach + wave,0.040,0.011
GAM: beach + wave,0.004,-0.001
RF: beach + wave,0.017,-0.011
GBT: beach + wave,0.009,-0.005


### 10b. Cross-validation (80%) vs. locked test (20%)

In [14]:
q2_results.set_index(["test set", "model"]).round(3)

CV R²  CV MAE (m)  Test R²  \
test set              model                                                   
A: 22 random sections Baseline: overall average -0.013       8.073   -0.010   
                      Linear: beach             -0.002       8.008    0.037   
                      GAM: beach                 0.028       7.864    0.090   
                      RF: beach                 -0.046       8.215   -0.013   
                      GBT: beach                -0.035       8.140   -0.002   
                      Linear: beach + wave       0.010       8.009    0.040   
                      GAM: beach + wave          0.033       7.907    0.004   
                      RF: beach + wave          -0.029       8.207    0.017   
                      GBT: beach + wave         -0.039       8.235    0.009   
B: every 5th section  Baseline: overall average -0.011       7.913   -0.000   
                      Linear: beach              0.028       7.722   -0.034   
                      GAM: beach                 0.070       7.516   -0.017   
                      RF: beach                 -0.015       7.879   -0.077   
                      GBT: beach                -0.028       7.920   -0.045   
                      Linear: beach + wave       0.043       7.687    0.011   
                      GAM: beach + wave          0.054       7.632   -0.001   
                      RF: beach + wave           0.003       7.887   -0.011   
                      GBT: beach + wave         -0.006       7.925   -0.005   

                                                 Test MAE (m)  
test set              model                                    
A: 22 random sections Baseline: overall average         7.760  
                      Linear: beach                     7.596  
                      GAM: beach                        7.362  
                      RF: beach                         7.773  
                      GBT: beach                        7.733  
                      Linear: beach + wave              7.612  
                      GAM: beach + wave                 7.661  
                      RF: beach + wave                  7.681  
                      GBT: beach + wave                 7.756  
B: every 5th section  Baseline: overall average         8.429  
                      Linear: beach                     8.622  
                      GAM: beach                        8.508  
                      RF: beach                         8.804  
                      GBT: beach                        8.624  
                      Linear: beach + wave              8.475  
                      GAM: beach + wave                 8.487  
                      RF: beach + wave                  8.530  
                      GBT: beach + wave                 8.463

---
## Notes

- **No data leakage:** each test set is removed before cross-validation, training and tuning.
  Baselines are computed from the training data only.
- **Same methodology as the main notebook,** except for the locked test sets: same models, features, settings,
  outlier rule, leave-one-winter-out (Q1) and 10-fold grouped CV (Q2), with GBT tuning inside each fold.
- **Several test sets:** a single locked test set can be easier or harder than average.
  If the conclusions hold across all of them, they don't depend on which data we held out.
- **Saved models for the demo:** `models/locked_test/q1_last_4_winters/` and `models/locked_test/q2_22_random_sections/`.
  They were trained on the 80% only, so predicting `test_rows.csv` reproduces the "Test" scores above.
  The baseline and "transect average + …" models use classes defined in this notebook (`TransectAverage`,
  `TransectAveragePlus`); to load those `.pkl` files elsewhere, define the same classes first.
- **Test set A for Question 1 locks away 2015–16** (the strongest El Niño), so the models train without it.
  That is a realistic check: a new winter can be stronger than anything in the training data.